# NB_02_A_MEASUREMENT_SPECIFICATION

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkthoughts/sensors-becker/blob/main/notebooks/NB_02_A_MEASUREMENT_SPECIFICATION.ipynb)

This notebook introduces the repository's next (02) engineering artifact bundle.


In [ ]:
NOTEBOOK_ID = "NB_02_A_MEASUREMENT_SPECIFICATION"
NOTEBOOK_FILENAME = f"{NOTEBOOK_ID}.ipynb"
NOTEBOOK_VERSION = "0.3.0"
ENGINEERING_STAGE = "A"
RELEASE_FILENAME = f"{NOTEBOOK_ID}.zip"

{
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "engineering_stage": ENGINEERING_STAGE,
    "release_filename": RELEASE_FILENAME,
}


## Initialize Notebook Runtime

This operation prepares the repository environment used to generate the bundle.


In [ ]:
from __future__ import annotations

import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
COLAB_REPOSITORY_ROOT = Path("/content/sensors-becker")


def install_colab_repository() -> Path:
    """Clone and install the repository in a fresh Colab runtime."""

    if COLAB_REPOSITORY_ROOT.exists():
        shutil.rmtree(COLAB_REPOSITORY_ROOT)

    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(COLAB_REPOSITORY_ROOT)],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--editable",
            str(COLAB_REPOSITORY_ROOT),
        ],
        check=True,
    )

    src_dir = COLAB_REPOSITORY_ROOT / "src"
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

    importlib.invalidate_caches()
    os.chdir(COLAB_REPOSITORY_ROOT)
    return COLAB_REPOSITORY_ROOT


try:
    import sensors_becker
except ModuleNotFoundError:
    repository_root = install_colab_repository()
    import sensors_becker
else:
    repository_root = Path(sensors_becker.__file__).resolve().parents[2]


if not (repository_root / "pyproject.toml").exists():
    raise FileNotFoundError(
        f"Repository root does not contain pyproject.toml: {repository_root}"
    )

from sensors_becker import initialize_notebook

runtime = initialize_notebook(
    start=repository_root,
    environment=(
        "google-colab"
        if repository_root == COLAB_REPOSITORY_ROOT
        else "repository-runtime"
    ),
)
context = runtime.context

print(f"Environment: {runtime.environment}")
print(f"Package: {Path(sensors_becker.__file__).resolve()}")
print(f"Repository root: {runtime.repository_root}")


## Validate Engineering Context

This operation validates the repository engineering context before artifact generation.


In [ ]:
runtime.validate()
print("Engineering context validation: PASSED")


## Prepare Notebook Bundle

This operation prepares the portable bundle directory and imports Notebook Dialogue Renderer v3.


In [ ]:
from __future__ import annotations

import json
import shutil
import zipfile
from datetime import date
from hashlib import sha256
from pathlib import Path

from IPython.display import Image, Markdown, display

from sensors_becker.dialogue_renderer import (
    DialogueFigure,
    DialogueNode,
    DialogueRelation,
    NotebookDialogueRenderer,
)

FOOTER = "Admissible generalizations trail leading specifications."
SUBTITLE = "Toward next-generation microcalorimeters."

renderer = NotebookDialogueRenderer(
    figsize=(12, 8),
    dpi=180,
    validate_layout=True,
)


def display_artifact(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Artifact does not exist: {path}")

    suffix = path.suffix.lower()

    if suffix == ".png":
        display(Image(filename=str(path)))
    elif suffix == ".md":
        display(Markdown(path.read_text(encoding="utf-8")))
    else:
        print(path.read_text(encoding="utf-8"))

    print(f"Artifact: {path.name}")


def sha256_for_path(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def two_step_figure(
    *,
    title: str,
    first_label: str,
    second_label: str,
    supporting_context: tuple[str, ...],
) -> DialogueFigure:
    """Return the canonical NB_02 dialogue scaffold using renderer v3."""

    return DialogueFigure(
        title=title,
        subtitle=SUBTITLE,
        footer=FOOTER,
        primary_nodes=(
            DialogueNode(
                first_label,
                x=0.5,
                y=0.67,
                width=0.42,
                height=0.10,
                role="input",
                fontsize=18,
            ),
            DialogueNode(
                second_label,
                x=0.5,
                y=0.46,
                width=0.42,
                height=0.12,
                role="primary",
                emphasis=True,
                fontsize=21,
            ),
        ),
        primary_relations=(
            DialogueRelation(
                start=(0.5, 0.62),
                end=(0.5, 0.52),
                role="input",
                directional=True,
                linewidth=2.0,
            ),
        ),
        supporting_context=supporting_context,
    )


def render_measurement_specification_trail(output_path: Path) -> Path:
    return renderer.render(
        two_step_figure(
            title="Measurement Specification Trail: Microcalorimeters",
            first_label="Measured State",
            second_label="Specification",
            supporting_context=(
                "Thermometer",
                "Cryogenic Environment",
            ),
        ),
        output_path,
    )


def render_measurement_record_trail(output_path: Path) -> Path:
    return renderer.render(
        two_step_figure(
            title="Measurement Record Trail: Microcalorimeters",
            first_label="Measurement Specification",
            second_label="Measurement Record",
            supporting_context=(
                "Measured State",
                "Cryogenic Environment",
            ),
        ),
        output_path,
    )


def render_engineering_evidence_trail(output_path: Path) -> Path:
    return renderer.render(
        two_step_figure(
            title="Engineering Evidence Trail: Microcalorimeters",
            first_label="Measurement Record",
            second_label="Engineering Evidence",
            supporting_context=(
                "Measurement Specification",
                "Thermometer",
            ),
        ),
        output_path,
    )


def render_engineering_statement_trail(output_path: Path) -> Path:
    return renderer.render(
        two_step_figure(
            title="Engineering Statement Trail: Microcalorimeters",
            first_label="Engineering Evidence",
            second_label="Engineering Statement",
            supporting_context=(
                "Measurement Record",
                "Measurement Specification",
            ),
        ),
        output_path,
    )


bundle_directory = runtime.paths.outputs / "releases" / NOTEBOOK_ID
bundle_directory.mkdir(parents=True, exist_ok=True)

for prior_path in bundle_directory.iterdir():
    if prior_path.is_file():
        prior_path.unlink()

print(f"Renderer: {renderer.__class__.__name__}")
print("Renderer version: 3")
print("Supporting-context API: noun-only")
print("Layout validation: enabled")
print(f"Bundle directory: {runtime.relative_path(bundle_directory)}")


## First (A)

This notebook introduces

**02_A_measurement_specification_trail.png**


In [ ]:
artifact_a_png = bundle_directory / "02_A_measurement_specification_trail.png"
artifact_a_alt = bundle_directory / "02_A_measurement_specification_trail.alt.md"

render_measurement_specification_trail(artifact_a_png)

artifact_a_alt.write_text(
    """# Alt Text

**Measurement Specification Trail: Microcalorimeters.** Diagram showing a measured state leading to a specification. A thermometer and cryogenic environment appear as supporting engineering context. Subtitle: “Toward next-generation microcalorimeters.” Footer: “Admissible generalizations trail leading specifications.”
""",
    encoding="utf-8",
)

display_artifact(artifact_a_png)
display_artifact(artifact_a_alt)


## Second (B)

This notebook introduces

**02_B_measurement_record_trail.png**


In [ ]:
artifact_b_png = bundle_directory / "02_B_measurement_record_trail.png"
artifact_b_alt = bundle_directory / "02_B_measurement_record_trail.alt.md"

render_measurement_record_trail(artifact_b_png)

artifact_b_alt.write_text(
    """# Alt Text

**Measurement Record Trail: Microcalorimeters.** Diagram showing a measurement specification leading to a measurement record. The measured state and cryogenic environment appear as supporting engineering context. Subtitle: “Toward next-generation microcalorimeters.” Footer: “Admissible generalizations trail leading specifications.”
""",
    encoding="utf-8",
)

display_artifact(artifact_b_png)
display_artifact(artifact_b_alt)


## Third (C)

This notebook introduces

**02_C_engineering_evidence_trail.png**


In [ ]:
artifact_c_png = bundle_directory / "02_C_engineering_evidence_trail.png"
artifact_c_alt = bundle_directory / "02_C_engineering_evidence_trail.alt.md"

render_engineering_evidence_trail(artifact_c_png)

artifact_c_alt.write_text(
    """# Alt Text

**Engineering Evidence Trail: Microcalorimeters.** Diagram showing a measurement record leading to engineering evidence. Measurement specification and thermometer appear as supporting engineering context. Subtitle: “Toward next-generation microcalorimeters.” Footer: “Admissible generalizations trail leading specifications.”
""",
    encoding="utf-8",
)

display_artifact(artifact_c_png)
display_artifact(artifact_c_alt)


## Fourth (D)

This notebook introduces

**02_D_engineering_statement_trail.png**


In [ ]:
artifact_d_png = bundle_directory / "02_D_engineering_statement_trail.png"
artifact_d_alt = bundle_directory / "02_D_engineering_statement_trail.alt.md"

render_engineering_statement_trail(artifact_d_png)

artifact_d_alt.write_text(
    """# Alt Text

**Engineering Statement Trail: Microcalorimeters.** Diagram showing engineering evidence leading to an engineering statement. Measurement record and measurement specification appear as supporting engineering context. Subtitle: “Toward next-generation microcalorimeters.” Footer: “Admissible generalizations trail leading specifications.”
""",
    encoding="utf-8",
)

display_artifact(artifact_d_png)
display_artifact(artifact_d_alt)


## Record Notebook Bundle

The following records identify and verify the portable artifact bundle.


In [ ]:
artifact_e_path = bundle_directory / "02_E_README.md"
artifact_f_path = bundle_directory / "02_F_notebook_metadata.json"
artifact_g_path = bundle_directory / "02_G_manifest.json"

concept_records = [
    {
        "artifact_order": "A",
        "artifact_id": "02_A_measurement_specification_trail",
        "engineering_concept": "Measurement Specification",
        "realizations": [artifact_a_png, artifact_a_alt],
    },
    {
        "artifact_order": "B",
        "artifact_id": "02_B_measurement_record_trail",
        "engineering_concept": "Measurement Record",
        "realizations": [artifact_b_png, artifact_b_alt],
    },
    {
        "artifact_order": "C",
        "artifact_id": "02_C_engineering_evidence_trail",
        "engineering_concept": "Engineering Evidence",
        "realizations": [artifact_c_png, artifact_c_alt],
    },
    {
        "artifact_order": "D",
        "artifact_id": "02_D_engineering_statement_trail",
        "engineering_concept": "Engineering Statement",
        "realizations": [artifact_d_png, artifact_d_alt],
    },
]

primary_artifact_paths = [
    realization
    for record in concept_records
    for realization in record["realizations"]
]

bundle_readme = f"""# {NOTEBOOK_ID}

This bundle contains the notebook's admitted engineering concepts in portable reading order.

## Engineering Dialogue

Measurement Specification

↓

Measurement Record

↓

Engineering Evidence

↓

Engineering Statement

## Engineering Artifacts

""" + "\n".join(
    f"- {record['artifact_order']}: {record['artifact_id']}"
    for record in concept_records
) + f"""

## Repository

{context.repository}

## Engineering Object

Microcalorimeter

## Engineering Direction

Toward next-generation microcalorimeters.

---

*{FOOTER}*
"""

artifact_e_path.write_text(bundle_readme, encoding="utf-8")

release_metadata = {
    "artifact_type": "notebook_bundle",
    "notebook_id": NOTEBOOK_ID,
    "notebook_filename": NOTEBOOK_FILENAME,
    "notebook_version": NOTEBOOK_VERSION,
    "repository": context.repository,
    "engineering_stage": ENGINEERING_STAGE,
    "engineering_scope": "measurement_specification",
    "engineering_object": "microcalorimeter",
    "runtime_environment": runtime.environment,
    "generated": date.today().isoformat(),
}

artifact_f_path.write_text(
    json.dumps(release_metadata, indent=2) + "\n",
    encoding="utf-8",
)

manifest_records = []

for concept in concept_records:
    for realization_order, path in enumerate(concept["realizations"], start=1):
        manifest_records.append(
            {
                "artifact_order": concept["artifact_order"],
                "artifact_id": concept["artifact_id"],
                "engineering_concept": concept["engineering_concept"],
                "realization_order": realization_order,
                "realization_type": (
                    "png" if path.suffix.lower() == ".png" else "alt_text"
                ),
                "filename": path.name,
                "size_bytes": path.stat().st_size,
                "sha256": sha256_for_path(path),
                "status": "verified",
            }
        )

for support_order, path in enumerate([artifact_e_path, artifact_f_path], start=1):
    manifest_records.append(
        {
            "artifact_order": path.name.split("_", 2)[1],
            "artifact_id": path.stem,
            "engineering_concept": "bundle_record",
            "realization_order": support_order,
            "realization_type": path.suffix.lower().lstrip("."),
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_for_path(path),
            "status": "verified",
        }
    )

manifest = {
    "release_filename": RELEASE_FILENAME,
    "repository": context.repository,
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "engineering_stage": ENGINEERING_STAGE,
    "generated": date.today().isoformat(),
    "engineering_dialogue": [
        "Measurement Specification",
        "Measurement Record",
        "Engineering Evidence",
        "Engineering Statement",
    ],
    "artifacts": manifest_records,
}

artifact_g_path.write_text(
    json.dumps(manifest, indent=2) + "\n",
    encoding="utf-8",
)

for path in (artifact_e_path, artifact_f_path, artifact_g_path):
    display_artifact(path)


## Verify Notebook Bundle

This operation verifies the complete nested artifact-and-realization sequence.


In [ ]:
bundle_artifact_paths = primary_artifact_paths + [
    artifact_e_path,
    artifact_f_path,
    artifact_g_path,
]

expected_names = [
    "02_A_measurement_specification_trail.png",
    "02_A_measurement_specification_trail.alt.md",
    "02_B_measurement_record_trail.png",
    "02_B_measurement_record_trail.alt.md",
    "02_C_engineering_evidence_trail.png",
    "02_C_engineering_evidence_trail.alt.md",
    "02_D_engineering_statement_trail.png",
    "02_D_engineering_statement_trail.alt.md",
    "02_E_README.md",
    "02_F_notebook_metadata.json",
    "02_G_manifest.json",
]

actual_names = [path.name for path in bundle_artifact_paths]

assert actual_names == expected_names
assert all(path.exists() and path.stat().st_size > 0 for path in bundle_artifact_paths)

with artifact_g_path.open(encoding="utf-8") as handle:
    manifest_check = json.load(handle)

for record in manifest_check["artifacts"]:
    path = bundle_directory / record["filename"]
    assert path.exists()
    assert sha256_for_path(path) == record["sha256"]

for path in bundle_artifact_paths:
    print(f"✓ {path.name} ({path.stat().st_size} bytes)")

assert manifest_check["engineering_dialogue"] == [
    "Measurement Specification",
    "Measurement Record",
    "Engineering Evidence",
    "Engineering Statement",
]

print("✓ Engineering dialogue order verified")
print("✓ Renderer v3 supporting-context layout completed")


## Package Notebook Bundle

This operation packages the verified artifacts without changing their portable reading order.


In [ ]:
release_directory = runtime.paths.outputs / "releases"
release_directory.mkdir(parents=True, exist_ok=True)
release_path = release_directory / RELEASE_FILENAME

with zipfile.ZipFile(release_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_artifact_paths:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(release_path) as archive:
    assert archive.namelist() == expected_names

assert release_path.stat().st_size > 0

print(
    f"✓ {runtime.relative_path(release_path)} "
    f"({release_path.stat().st_size} bytes)"
)


## Download Notebook Bundle

In Google Colab, this operation downloads the ZIP. Elsewhere, it prints the saved path.


In [ ]:
if runtime.environment == "google-colab":
    from google.colab import files
    files.download(str(release_path))
else:
    print(release_path)


*Admissible generalizations trail leading specifications.*
